# Data Wrangling Exercise: Cleaning a Messy Compound-Screening Dataset

**Scenario.** A screening lab measured physicochemical properties and assay conditions for ~1,500 small
molecules and recorded whether each compound was *active* in a biochemical assay. The data were collected by
several people over ten months, exported from different instruments and typed into a shared spreadsheet.
You now want to train a model that predicts `active` from the measured properties - but first the data
need to be cleaned.

**Learning goals**
1. Explore an unknown dataset (EDA) and *find* the problems before fixing them
2. Standardise column names and categorical labels
3. Handle missing-value codes, numbers stored as text, and censored values (`<LOD`)
4. Convert units (°C / K / °F and µM / mM / mg/mL)
5. Create boolean **flag columns** instead of silently deleting information
6. Detect and treat outliers and duplicates
7. Measure what all of this does to a **Random Forest** classifier

**Hint:** Work through the parts in order; each part ends with a checkpoint cell (`assert`s)
that tells you whether you are on track.

**Raw columns**

| Raw column | Meaning |
|---|---|
| `Compound ID` | identifier of the compound |
| `MW (g/mol)`, `LogP`, `tPSA [A^2]`, `H-Bond Donors` | physicochemical properties |
| `Assay pH`, `Temp` + `Temp_unit` | assay conditions |
| `Solubility` + `Solubilty unit` | measured solubility |
| `Method`, `operator `, `PLATE`, `Measurement Date`, `Project` | metadata |
| `Notes` | free text from the operator |
| `Active?` | **target**: was the compound active in the assay? |

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedGroupKFold, cross_val_score

pd.set_option("display.max_columns", 30)
plt.rcParams["figure.dpi"] = 100

raw = pd.read_csv("compound_screening_raw.csv")   # never modify `raw` - always work on a copy

## Part 1 – First look (EDA)

Before cleaning anything, find out what is wrong. Work through the cells and **write down every problem you
find** (a list in a markdown cell is fine). There are at least 10 distinct problems in this dataset.

In [3]:
# 1.1  Shape, dtypes, a few random rows  (hint: .shape, .dtypes, .sample(10))
# TODO
df=raw.copy()
print(df.shape)
print(df.dtypes)
print(df.sample(15))    

(1635, 17)
Compound ID             str
MW (g/mol)              str
LogP                    str
tPSA [A^2]              str
H-Bond Donors           str
Assay pH                str
Temp                float64
Temp_unit               str
Solubility              str
Solubilty unit          str
Method                  str
operator                str
PLATE                 int64
Measurement Date        str
Project                 str
Notes                   str
Active?                 str
dtype: object
     Compound ID MW (g/mol)   LogP tPSA [A^2] H-Bond Donors Assay pH    Temp  \
359     CPD-0382      471.3   4.45       83.7           3.0      7.4  297.82   
1063    CPD-0752      334.6   2.49      66.99           0.0      7.0   24.86   
1173    CPD-1308      464.8   0.11      162.7           1.0      7.4   20.60   
882     CPD-0907      340.8   2.98       34.9           1.0      7.4  297.87   
1066    CPD-0029      395.1  -0.18       76.4           0.0      7.0   19.85   
91      CPD-0988   

In [6]:
# 1.2  Which columns that *should* be numeric are stored as text? Which unusual tokens
#      ("NA", "n.a.", "-", ...) are hiding in them?
#      Hint: pd.to_numeric(col, errors="coerce") gives NaN where conversion fails;
#            then look at .value_counts() of the entries that failed.
# TODO
for col in df.columns:
    converted = pd.to_numeric(df[col], errors="coerce")
    failed = df[col][converted.isna() & df[col].notna()]
    print(failed.value_counts())

Compound ID
CPD-0424    3
CPD-0622    3
CPD-0452    3
CPD-1420    3
CPD-0608    2
           ..
CPD-1131    1
CPD-1295    1
CPD-0861    1
CPD-1460    1
CPD-1127    1
Name: count, Length: 1500, dtype: int64
MW (g/mol)
n.a.           7
234.2 g/mol    3
-              3
348.9 g/mol    2
309.5 g/mol    2
              ..
441.6 g/mol    1
303.8 g/mol    1
271.5 g/mol    1
276.5 g/mol    1
510.6 g/mol    1
Name: count, Length: 116, dtype: int64
LogP
n.a.    12
-        6
Name: count, dtype: int64
tPSA [A^2]
n.a.    14
-        8
Name: count, dtype: int64
H-Bond Donors
-       6
n.a.    3
Name: count, dtype: int64
Assay pH
-       5
n.a.    3
Name: count, dtype: int64
Series([], Name: count, dtype: int64)
Temp_unit
C         743
°C        347
kelvin    125
°F        119
K         117
c          98
F          86
Name: count, dtype: int64
Solubility
<LOD    37
n.a.    13
-        2
Name: count, dtype: int64
Solubilty unit
mM       437
uM       404
mg/mL    393
µM       315
UM        86
Name: co

In [20]:
# 1.3  Which categorical columns have inconsistent spellings?
#      Hint: value_counts() on Method, 'operator ', Active?, Temp_unit, 'Solubilty unit'
# TODO

for col in df.columns: 
    if col in ["Method", "operator ", "Active?", "Temp_unit", "Solubilty unit"]:
        print(f"Column: {col}")
        print(df[col].value_counts())
    else: 
        continue

Column: Temp_unit
Temp_unit
C         743
°C        347
kelvin    125
°F        119
K         117
c          98
F          86
Name: count, dtype: int64
Column: Solubilty unit
Solubilty unit
mM       437
uM       404
mg/mL    393
µM       315
UM        86
Name: count, dtype: int64
Column: Method
Method
FP                           221
fp                           198
Fluorescence Polarization    193
FP                           180
TR-FRET                      139
tr-fret                      117
Alphascreen                  115
AlphaScreen                  114
TR-FRET                      110
TRFRET                       108
alpha screen                  99
n.a.                           9
-999                           8
-                              3
Name: count, dtype: int64
Column: operator 
operator 
Ben Meier        323
Anna Keller      302
David Huber      301
Chiara Rossi     285
chiara rossi      45
ben meier         40
DAVID HUBER       40
anna keller       37
BEN MEIER    

In [21]:
# 1.4  Summary statistics of the numeric columns. Which values are physically impossible?
#      Which columns are constant / useless? Are there duplicated rows or compound IDs?
# TODO
for col in df.columns: 
    if col in ["MW (g/mol)","LogP", "tPSA [A^2]", "H-Bond Donors" ,"Assay pH", "Temp", "Solubility"]: 
        numeric_col= pd.to_numeric(df[col], errors="coerce")
        print (f"Column:{col}")
        print (numeric_col.describe())
    else: 
        continue

Column:MW (g/mol)
count     1466.000000
mean       444.331719
std       1812.918888
min       -999.000000
25%        299.300000
50%        360.300000
75%        413.175000
max      35000.000000
Name: MW (g/mol), dtype: float64
Column:LogP
count    1578.000000
mean       -5.577662
std        90.747394
min     -1003.460000
25%         1.650000
50%         2.590000
75%         3.460000
max        99.900000
Name: LogP, dtype: float64
Column:tPSA [A^2]
count    1567.000000
mean       75.063695
std       121.673064
min      -999.000000
25%        64.150000
50%        84.100000
75%       105.250000
max       900.000000
Name: tPSA [A^2], dtype: float64
Column:H-Bond Donors
count    1584.000000
mean       -2.673611
std        66.413121
min      -999.000000
25%         1.000000
50%         2.000000
75%         3.000000
max         6.000000
Name: H-Bond Donors, dtype: float64
Column:Assay pH
count    1606.000000
mean        2.858842
std        66.309577
min      -999.000000
25%         7.000000
5

**Your problem list** (double-click to edit):

1. Hiding tokens : (n.a., and "-" in several columns,  900.0 g/mol in MW, )
2. Inconsistent units (solubility_unit, temp_unit)
3. Inconsistent formats (method, operator, active?) 
4. string types if it should be numerical 
5. Min value of MW is negative (-999)
6. Max value of MW seems to high (wrong unit)
7. LogP: min and max value seem off. 
8. tPSA: min and max value seem off (-999 , 900)
9. H-Bond donors: min value -999 is wrong. 
10. Assay pH: min -999 is wrong 
11. solubility and temp must be corrected for unit to compare them. (but also impossible values like -999)


## Part 2 – Standardise column names

Target format: `snake_case`, lowercase, no units, no spaces/brackets/special characters, no typos.
Do it **programmatically** (string methods / regex on `df.columns`), not by retyping 17 names by hand –
then fix the remaining special cases with `rename`.

Final names we want: `compound_id, mw, logp, tpsa, hbd, ph, temp_value, temp_unit, solubility, solubility_unit,
method, operator, plate, measurement_date, project, notes, is_active`

In [ ]:
df = raw.copy()

# TODO: 1) lowercase + strip whitespace   2) remove "(...)" and "[...]"   3) replace every run of
#       non-alphanumeric characters by "_"   4) strip leading/trailing "_"   5) rename special cases
#       (h_bond_donors -> hbd, assay_ph -> ph, temp -> temp_value, solubilty_unit -> solubility_unit, active -> is_active)

TypeError: replace() got an unexpected keyword argument 'regex'

In [ ]:
# ---- Checkpoint 2 ----
expected = ["compound_id", "mw", "logp", "tpsa", "hbd", "ph", "temp_value", "temp_unit", "solubility",
            "solubility_unit", "method", "operator", "plate", "measurement_date", "project", "notes", "is_active"]
assert df.columns.tolist() == expected, f"Column names differ:\n{df.columns.tolist()}"
print("Part 2 OK")

## Part 3 – Missing values, numbers stored as text, censored values

1. Strip leading/trailing whitespace in all text columns.
2. Replace all missing-value codes (`NA`, `n.a.`, `N/A`, `-`, empty string, `-999`) by real `NaN`.
   *Think:* why is `-999` dangerous if you forget it?
3. `mw` contains entries like `"402.0 g/mol"` – extract the number.
4. Entries `<LOD` in `solubility` mean "below the limit of detection". Create a **boolean flag column**
   `is_below_lod` (True for these rows) **before** converting the column to numbers – the information
   "below LOD" is real information, and we do not want to lose it just because we turn the value into NaN.
5. Convert `mw, logp, tpsa, hbd, ph, solubility` to float.

Remove the `project` column (constant, no information) at the end of this part.

In [ ]:
# TODO: steps 1-5 above (hint: .str.strip(), .replace([...], np.nan), .str.replace(), pd.to_numeric(errors="coerce"))

df = df.drop(columns=["project"])

In [ ]:
# ---- Checkpoint 3 ----
for c in ["mw", "logp", "tpsa", "hbd", "ph", "solubility"]:
    assert pd.api.types.is_float_dtype(df[c]) or pd.api.types.is_integer_dtype(df[c]), f"{c} is not numeric"
assert "project" not in df.columns
assert df["is_below_lod"].dtype == bool and 20 <= df["is_below_lod"].sum() <= 55
assert (df[["mw", "logp", "tpsa", "hbd", "ph"]] == -999).sum().sum() == 0
print("Part 3 OK")

## Part 4 – Categorical columns and dates

* `method` → exactly three categories: `FP`, `TR-FRET`, `AlphaScreen`
* `operator` → title case, no stray spaces (four people)
* `is_active` → integer 0/1 (`active`, `Active`, `1`, `yes`, … → 1)
* `measurement_date` → real `datetime`. The dates come in three formats (`2024-03-05`, `05.03.2024`,
  `March 05, 2024`); the dotted ones are day-first (European). Dates are a classic trap: a wrong format
  guess does not raise an error, it silently gives you wrong dates. All measurements were taken between
  2024-01-08 and 2024-11-02 – use this to check your result!

In [ ]:
# TODO
# hint for method: lowercase, remove everything that is not a letter, then map {"fp": "FP", ...}
# hint for dates: parse each of the three formats separately (format="%Y-%m-%d", ...; errors="coerce")
#                  and combine with .fillna(). Do NOT rely on format="mixed" - check why!

In [ ]:
# ---- Checkpoint 4 ----
assert set(df["method"].dropna().unique()) == {"FP", "TR-FRET", "AlphaScreen"}
assert df["operator"].nunique() == 4
assert set(df["is_active"].unique()) == {0, 1}
assert pd.api.types.is_datetime64_any_dtype(df["measurement_date"])
assert df["measurement_date"].notna().all(), "Some dates could not be parsed"
assert df["measurement_date"].min() >= pd.Timestamp("2024-01-08") and df["measurement_date"].max() <= pd.Timestamp("2024-11-02"),     "Some dates were parsed wrongly (day/month swapped?)"
print("Part 4 OK")

## Part 5 – Unit conversion

Create two harmonised columns:

* `temp_c` – temperature in **°C**   (K → °C: subtract 273.15; °F → °C: (F − 32) · 5/9)
* `solubility_um` – solubility in **µM**
  * mM → µM: × 1000
  * mg/mL → µM: needs the molecular weight!   µM = (mg/mL) / MW[g/mol] · 10⁶   (check this on paper!)

First normalise the unit strings (`°`, `µ`, upper/lower case, `kelvin`), then use `np.select` or `.map`.
**Careful:** a few `mw` values are physically impossible (you will flag them properly in Part 6). Make sure such
values do not contaminate the mg/mL → µM conversion – otherwise you create new garbage from old garbage.

Afterwards drop the old value/unit columns. Does the result look physically plausible?
(Assay temperatures in this lab are between roughly 20 and 38 °C.)

In [ ]:
# TODO: temp_c and solubility_um

# plausibility check - print .describe() of the new columns

In [ ]:
# ---- Checkpoint 5 ----
assert df["temp_c"].between(15, 40).all(), "Temperature conversion looks wrong"
assert df["solubility_um"].dropna().between(0.01, 1e5).all(), "Solubility conversion looks wrong"
print("Part 5 OK")

## Part 6 – Flags: quality issues, duplicates, outliers

Instead of deleting rows immediately, first add **boolean flag columns**. This keeps your decisions
transparent and reversible – you can compare "with" and "without" later (Part 8).

1. `qc_issue` – True if the free-text `notes` indicate an unreliable measurement
   (*contaminat…, precipitat…, pipetting error, hemoly…*; careful: spelling and capitalisation vary).
   Notes such as "repeat measurement" or "ok" are **not** quality issues.
2. `is_duplicate` – True for every repeated entry of a `compound_id` except the first one.
   (`df.duplicated(subset=...)`. Why is `df.duplicated()` without `subset` not enough?)
3. `is_outlier` – True if `mw`, `logp` or `tpsa` is outside a **physically sensible range**
   (suggestion: MW 50–1500, logP −5…10, tPSA 1–250). Set the impossible values themselves to `NaN`,
   because they are measurement errors, not real compounds.
4. `has_missing_feature` – True if any of the model features (`mw, logp, tpsa, hbd, ph, temp_c, solubility_um`)
   is missing *at this point* (after step 3).

**Discussion:** `solubility_um` has a long right tail. Would you flag the large values as outliers using the
IQR rule? Try it on the raw and on the log scale and decide. (Hint: solubility is roughly log-normal.)

In [ ]:
FEATURES = ["mw", "logp", "tpsa", "hbd", "ph", "temp_c", "solubility_um"]

# TODO: qc_issue, is_duplicate, is_outlier (+ set impossible values to NaN), has_missing_feature

In [ ]:
# Discussion cell: IQR rule on solubility_um, raw scale vs. log scale
# TODO

In [ ]:
# ---- Checkpoint 6 ----
for c in ["qc_issue", "is_duplicate", "is_outlier", "has_missing_feature"]:
    assert df[c].dtype == bool, f"{c} must be boolean"
assert 120 <= df["qc_issue"].sum() <= 190
assert 110 <= df["is_duplicate"].sum() <= 160
assert 15 <= df["is_outlier"].sum() <= 50
assert df["mw"].dropna().between(50, 1500).all() and df["tpsa"].dropna().between(1, 250).all()
print("Part 6 OK")

## Part 7 – Before / after: what did the cleaning change?

The two cells below are **given**. Run them and answer the questions underneath.

In [ ]:
def plot_distributions(before, after, cols=("logp", "tpsa", "mw", "temp", "solubility")):
    fig, axes = plt.subplots(2, len(cols), figsize=(3.2 * len(cols), 5.5))
    for j, c in enumerate(cols):
        for i, (d, name, colr) in enumerate([(before, "BEFORE", "tab:red"), (after, "AFTER", "tab:blue")]):
            x = d[c].dropna()
            if c == "solubility":
                x = np.log10(x[x > 0])
            axes[i, j].hist(x, bins=40, color=colr, alpha=0.8)
            axes[i, j].set_title(f"{name}: {c}" + (" (log10)" if c == "solubility" else ""), fontsize=9)
    plt.tight_layout(); plt.show()


def plot_patterns(before, after):
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    for k, (d, name) in enumerate([(before, "BEFORE"), (after, "AFTER")]):
        ax = axes[0, k]
        for lab, colr in [(0, "tab:grey"), (1, "tab:orange")]:
            s = d[d["active"] == lab]
            ax.scatter(s["logp"], s["tpsa"], s=8, alpha=0.5, c=colr, label=f"active={lab}")
        ax.set_xlabel("logP"); ax.set_ylabel("tPSA"); ax.set_title(f"{name}: logP vs. tPSA"); ax.legend()
        ax = axes[1, k]
        sol = d["solubility"].where(d["solubility"] > 0)
        data = [np.log10(sol[d["active"] == lab].dropna()) for lab in (0, 1)]
        ax.boxplot(data, tick_labels=["inactive", "active"])
        ax.set_ylabel("log10 solubility (values as recorded)" if k == 0 else "log10 solubility [µM]")
        ax.set_title(f"{name}: solubility by class")
    plt.tight_layout(); plt.show()


before = pd.DataFrame({
    "logp": pd.to_numeric(raw["LogP"], errors="coerce"),
    "tpsa": pd.to_numeric(raw["tPSA [A^2]"], errors="coerce"),
    "mw": pd.to_numeric(raw["MW (g/mol)"].astype(str).str.replace("g/mol", "", regex=False), errors="coerce"),
    "temp": raw["Temp"],
    "solubility": pd.to_numeric(raw["Solubility"], errors="coerce"),
    "active": raw["Active?"].astype(str).str.strip().str.lower().isin(["active", "1", "yes"]).astype(int),
})
after = df.rename(columns={"temp_c": "temp", "solubility_um": "solubility", "is_active": "active"})

plot_distributions(before, after)
plot_patterns(before, after)

**Questions**

1. Which axes changed the most between BEFORE and AFTER? Which single problem caused this?
2. In the logP-vs-tPSA scatter plot: what pattern separates active from inactive compounds, and why was it
   hard to see before?
3. In the solubility boxplot: why do active and inactive compounds look identical BEFORE, and different AFTER?
4. The temperature histogram BEFORE shows several clusters. Which values belong to which unit?
5. Is a value of `logP = 6.5` an outlier that should be removed? How did we decide the limits in Part 6?

## Part 8 – Does the cleaning improve a Random Forest?

**Task:** predict `is_active` from the measured properties.

The helper below evaluates a Random Forest with 5-fold cross-validation, repeated 3 times. It reports the
mean accuracy in two ways:

* **random CV** – rows are split randomly (what most people do by default)
* **grouped CV** – all rows of the same compound end up in the same fold, so a compound is never in the
  training *and* the test set

Missing values are imputed (median) *inside* the pipeline, i.e. separately in each training fold.

In [ ]:
def evaluate(X, y, groups, n_estimators=200, repeats=3):
    """Returns (accuracy with random CV, accuracy with grouped CV), each as mean over folds and repeats."""
    model = make_pipeline(SimpleImputer(strategy="median"),
                          RandomForestClassifier(n_estimators, random_state=0, n_jobs=-1))
    rnd = cross_val_score(model, X, y, cv=RepeatedStratifiedKFold(n_splits=5, n_repeats=repeats, random_state=0))
    grp = []
    for r in range(repeats):
        cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=r)
        grp.append(cross_val_score(model, X, y, cv=cv, groups=groups).mean())
    return rnd.mean(), float(np.mean(grp))

### 8.1 Baseline: the 'quick and dirty' pipeline on the raw data

In [ ]:
# Quick-and-dirty: drop ID/notes/date, convert whatever can be converted, one-hot encode the rest
X_raw = raw.drop(columns=["Compound ID", "Notes", "Measurement Date", "Project", "Active?"]).copy()
for c in X_raw.columns:
    conv = pd.to_numeric(X_raw[c], errors="coerce")
    if conv.notna().mean() > 0.5:
        X_raw[c] = conv
X_raw = pd.get_dummies(X_raw)
y_raw = raw["Active?"].astype(str).str.strip().str.lower().isin(["active", "1", "yes"]).astype(int)

acc_raw = evaluate(X_raw, y_raw, groups=raw["Compound ID"])
print(f"RAW data   ->  {X_raw.shape[1]} features | random CV: {acc_raw[0]:.3f} | grouped CV: {acc_raw[1]:.3f}")

**Before you continue:** how many features did `get_dummies` create from the raw data and why?
Which features did the Random Forest "see" for temperature and solubility?

### 8.2 Your cleaned data – three scenarios

Build the feature matrix from `df`: the seven numeric `FEATURES` plus the one-hot encoded `method`
(`pd.get_dummies(df["method"], dtype=float)`). Then evaluate three versions of the training data:

* **A** – cleaned columns and units, but keep duplicates and QC-flagged rows
* **B** – A, but remove duplicates (`is_duplicate`)
* **C** – B, but also remove rows with a QC issue (`qc_issue`)

Define each scenario as a boolean row mask (this is exactly what the flag columns are for!).

In [ ]:
X_all = pd.concat([df[FEATURES], pd.get_dummies(df["method"], dtype=float)], axis=1)
y_all = df["is_active"]

# TODO: define the three masks and fill the results dict
masks = {
    "A: cleaned, keep duplicates + QC rows": ...,
    "B: A without duplicates": ...,
    "C: B without QC-flagged rows": ...,
}

results = {"RAW (quick and dirty)": acc_raw}
for name, m in masks.items():
    results[name] = evaluate(X_all[m], y_all[m], groups=df.loc[m, "compound_id"])

res = pd.DataFrame(results, index=["random CV", "grouped CV"]).T.round(3)
res

In [ ]:
res.plot.barh(figsize=(8, 3.5)); plt.xlim(0.6, 0.95); plt.xlabel("accuracy"); plt.tight_layout(); plt.show()

**Questions**

1. Compare RAW and B in the *grouped CV* column. How many accuracy points did the cleaning of units, glitches and
   categories buy you?
2. Compare A and B in the *random CV* column: the accuracy goes **down** when duplicates are removed. Is B worse?
   Which of the two numbers can you trust, and why? (Hint: what happens when a compound appears in the training
   *and* the test fold?)
3. What does removing the QC-flagged rows (C) improve, and what is the price? Would you also remove such rows from
   the *test* set in a real project?
4. Re-run the notebook after changing the outlier limits in Part 6 (e.g. logP −1…7). What happens?

### 8.3 Feature importance before vs. after

In [ ]:


def top_importances(X, y, k=8):
    X = pd.DataFrame(SimpleImputer(strategy="median").fit_transform(X), columns=X.columns) # RF cannot handle NaNs - fill with median

    # TODO: fit a Random Forest on (a) the RAW features and (b) scenario C and plot the 8 most important features side by side
    #       (hint: model.feature_importances_, pd.Series(..., index=X.columns).nlargest(8))

# plotting
mC = masks["C: B without QC-flagged rows"]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
top_importances(X_raw, y_raw).sort_values().plot.barh(ax=axes[0], color="tab:red", title="RAW data")
top_importances(X_all[mC], y_all[mC]).sort_values().plot.barh(ax=axes[1], color="tab:blue", title="Cleaned (C)")
plt.tight_layout(); plt.show()

**Questions:** Which features does the model rely on in the raw data (hint: look at `Temp`, `Solubility`, `PLATE`)?
Are the same features important after cleaning? Is a high importance for `PLATE` or the compound ID a good sign?

## Part 9 – Bonus challenges

1. **Impute vs. drop.** Compare dropping all rows with missing features to median imputation, and to
   imputation + the `has_missing_feature` flag as an additional feature.
2. **Flags as features.** Add `is_below_lod` as a feature. Does it help? Why might "below LOD" be informative?
3. **Re-measurements.** Instead of keeping only the first entry per compound, average the numeric columns of
   repeated measurements (`groupby("compound_id")`). Does the accuracy change?
4. **Pipeline.** Turn Parts 2–6 into one function `clean(raw) -> df` and test it on a second dataset:
   `python generate_messy_data.py 7` writes a new file with a different random seed. Does your function
   still work without changes? (What did you hard-code?)
5. **Honest evaluation.** Hold out 20 % of the compounds *before* any cleaning decision that uses the labels
   (e.g. QC removal) and report the final accuracy only once on this untouched test set.